# Handwritten Digit Recognition - Model Training

This notebook trains CNN and baseline models on the MNIST dataset for digit recognition (0-9).

**Models:**
- Baseline ANN (Fully Connected Network)
- Enhanced CNN with Batch Normalization and Dropout

## 1. Setup and Imports

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Conv2D, MaxPooling2D, Dense, Dropout, Flatten,
    InputLayer, BatchNormalization
)
from tensorflow.keras.regularizers import l2
from tensorflow.keras.datasets import mnist
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import (
    ModelCheckpoint, EarlyStopping, ReduceLROnPlateau
)
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report

# Set style
plt.style.use('dark_background')
sns.set_palette('husl')

print(f'TensorFlow version: {tf.__version__}')
print(f'GPU Available: {len(tf.config.list_physical_devices("GPU")) > 0}')

## 2. Configuration

In [ ]:
# Training Configuration
CONFIG = {
    'batch_size': 128,
    'epochs': 30,
    'validation_split': 0.2,
    'use_augmentation': True,
    'checkpoint_dir': '../backend/checkpoints',
}

# Create checkpoint directory if needed
os.makedirs(CONFIG['checkpoint_dir'], exist_ok=True)

## 3. Load and Preprocess Data

In [ ]:
# Load MNIST
(X_train_full, y_train_full), (X_test, y_test) = mnist.load_data()

print(f'Training samples: {X_train_full.shape[0]}')
print(f'Test samples: {X_test.shape[0]}')
print(f'Image shape: {X_train_full.shape[1:]}')

In [ ]:
# Normalize and reshape
X_train_full = X_train_full.astype('float32') / 255.0
X_test = X_test.astype('float32') / 255.0

X_train_full = X_train_full.reshape(-1, 28, 28, 1)
X_test = X_test.reshape(-1, 28, 28, 1)

# One-hot encode
y_train_full_cat = to_categorical(y_train_full, 10)
y_test_cat = to_categorical(y_test, 10)

# Split into train/validation
X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full_cat,
    test_size=CONFIG['validation_split'],
    random_state=42,
    stratify=y_train_full
)

print(f'Training: {X_train.shape[0]}')
print(f'Validation: {X_val.shape[0]}')
print(f'Test: {X_test.shape[0]}')

## 4. Define Models

In [ ]:
def create_baseline_model():
    """Simple fully-connected neural network."""
    model = Sequential([
        InputLayer(input_shape=(28, 28, 1)),
        Flatten(),
        Dense(128, activation='relu'),
        Dropout(0.2),
        Dense(64, activation='relu'),
        Dropout(0.2),
        Dense(10, activation='softmax')
    ], name='baseline_ann')
    
    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    return model

In [ ]:
def create_cnn_model():
    """Enhanced CNN with BatchNorm and Dropout."""
    model = Sequential([
        InputLayer(input_shape=(28, 28, 1)),
        
        # Block 1
        Conv2D(32, (3, 3), activation='relu', padding='same'),
        BatchNormalization(),
        Conv2D(32, (3, 3), activation='relu', padding='same'),
        BatchNormalization(),
        MaxPooling2D((2, 2)),
        Dropout(0.25),
        
        # Block 2
        Conv2D(64, (3, 3), activation='relu', padding='same'),
        BatchNormalization(),
        Conv2D(64, (3, 3), activation='relu', padding='same'),
        BatchNormalization(),
        MaxPooling2D((2, 2)),
        Dropout(0.25),
        
        # Dense layers
        Flatten(),
        Dense(256, activation='relu', kernel_regularizer=l2(0.001)),
        BatchNormalization(),
        Dropout(0.5),
        Dense(128, activation='relu', kernel_regularizer=l2(0.001)),
        Dropout(0.5),
        Dense(10, activation='softmax')
    ], name='enhanced_cnn')
    
    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    return model

## 5. Data Augmentation

In [ ]:
datagen = ImageDataGenerator(
    rotation_range=10,
    width_shift_range=0.1,
    height_shift_range=0.1,
    zoom_range=0.1,
    shear_range=0.1,
    fill_mode='nearest'
)

# Visualize augmented samples
datagen.fit(X_train)

fig, axes = plt.subplots(2, 5, figsize=(12, 5))
fig.suptitle('Data Augmentation Examples', fontsize=14)

sample = X_train[0:1]
for i, ax in enumerate(axes.flat):
    aug_img = datagen.random_transform(sample[0])
    ax.imshow(aug_img.squeeze(), cmap='gray')
    ax.axis('off')

plt.tight_layout()
plt.show()

## 6. Training Callbacks

In [ ]:
def get_callbacks(model_name):
    return [
        ModelCheckpoint(
            filepath=os.path.join(CONFIG['checkpoint_dir'], f'{model_name}_best.keras'),
            monitor='val_accuracy',
            mode='max',
            save_best_only=True,
            verbose=1
        ),
        EarlyStopping(
            monitor='val_loss',
            patience=7,
            restore_best_weights=True,
            verbose=1
        ),
        ReduceLROnPlateau(
            monitor='val_loss',
            factor=0.5,
            patience=3,
            min_lr=1e-6,
            verbose=1
        )
    ]

## 7. Train Baseline Model

In [ ]:
baseline_model = create_baseline_model()
baseline_model.summary()

In [ ]:
print('Training Baseline Model...')
baseline_history = baseline_model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=CONFIG['epochs'],
    batch_size=CONFIG['batch_size'],
    callbacks=get_callbacks('baseline'),
    verbose=1
)

## 8. Train CNN Model

In [ ]:
cnn_model = create_cnn_model()
cnn_model.summary()

In [ ]:
print('Training CNN Model with Data Augmentation...')

cnn_history = cnn_model.fit(
    datagen.flow(X_train, y_train, batch_size=CONFIG['batch_size']),
    steps_per_epoch=len(X_train) // CONFIG['batch_size'],
    validation_data=(X_val, y_val),
    epochs=CONFIG['epochs'],
    callbacks=get_callbacks('cnn'),
    verbose=1
)

## 9. Training Curves

In [ ]:
def plot_history(history, title):
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Accuracy
    axes[0].plot(history.history['accuracy'], label='Train', linewidth=2)
    axes[0].plot(history.history['val_accuracy'], label='Val', linewidth=2, linestyle='--')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Accuracy')
    axes[0].set_title(f'{title} - Accuracy')
    axes[0].legend()
    axes[0].grid(alpha=0.3)
    
    # Loss
    axes[1].plot(history.history['loss'], label='Train', linewidth=2)
    axes[1].plot(history.history['val_loss'], label='Val', linewidth=2, linestyle='--')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Loss')
    axes[1].set_title(f'{title} - Loss')
    axes[1].legend()
    axes[1].grid(alpha=0.3)
    
    plt.tight_layout()
    plt.show()

plot_history(baseline_history, 'Baseline ANN')
plot_history(cnn_history, 'Enhanced CNN')

## 10. Evaluation

In [ ]:
# Evaluate both models
baseline_loss, baseline_acc = baseline_model.evaluate(X_test, y_test_cat, verbose=0)
cnn_loss, cnn_acc = cnn_model.evaluate(X_test, y_test_cat, verbose=0)

print('=' * 50)
print('TEST RESULTS')
print('=' * 50)
print(f'Baseline ANN: {baseline_acc*100:.2f}%')
print(f'Enhanced CNN: {cnn_acc*100:.2f}%')
print(f'Improvement: +{(cnn_acc - baseline_acc)*100:.2f}%')

In [ ]:
# Confusion Matrix for CNN
y_pred = cnn_model.predict(X_test, verbose=0)
y_pred_labels = np.argmax(y_pred, axis=1)

cm = confusion_matrix(y_test, y_pred_labels)

plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=range(10), yticklabels=range(10))
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('CNN Confusion Matrix')
plt.show()

In [ ]:
# Classification Report
print(classification_report(y_test, y_pred_labels, target_names=[f'Digit {i}' for i in range(10)]))

## 11. Sample Predictions

In [ ]:
# Show sample predictions
fig, axes = plt.subplots(3, 5, figsize=(12, 8))
fig.suptitle('Sample Predictions', fontsize=14)

indices = np.random.choice(len(X_test), 15, replace=False)

for i, ax in enumerate(axes.flat):
    idx = indices[i]
    ax.imshow(X_test[idx].squeeze(), cmap='gray')
    
    pred = y_pred_labels[idx]
    true = y_test[idx]
    conf = y_pred[idx, pred] * 100
    
    color = 'lime' if pred == true else 'red'
    ax.set_title(f'T:{true} P:{pred} ({conf:.1f}%)', color=color)
    ax.axis('off')

plt.tight_layout()
plt.show()

## 12. Save Final Models

In [ ]:
# Save final models
baseline_model.save(os.path.join(CONFIG['checkpoint_dir'], 'baseline_final.keras'))
cnn_model.save(os.path.join(CONFIG['checkpoint_dir'], 'cnn_final.keras'))

print(f"Models saved to {CONFIG['checkpoint_dir']}/")
print('- baseline_final.keras')
print('- cnn_final.keras')

## Summary

- Trained Baseline ANN and Enhanced CNN on MNIST
- Used data augmentation for better generalization
- CNN significantly outperforms baseline
- Models saved for inference API